In [1]:
import json
import os
from datetime import date, datetime, timedelta

import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv
from openai import OpenAI

# Course machines keep the key in llm_ref/openai_key.env (one line: OPENAI_API_KEY=sk-...).
# On Colab / your laptop: comment the line below and set the environment variable yourself,
# e.g.  os.environ["OPENAI_API_KEY"] = getpass.getpass("API key: ")   — never hard-code keys in a notebook you might share.
load_dotenv("/Users/shivam13juna/Documents/scaler/iitr_classes/llm_ref/openai_key.env")

assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY is not set — fix this before continuing."

client = OpenAI()

# Any Responses-API chat model works here. We use a small, cheap one: an agent makes
# MANY model calls per user request, so per-call price matters mimport os, json
from dotenv import load_dotenv

import textwrap



def pretty_print(*args):
    text = " ".join(str(arg) for arg in args)
    try:
        print(textwrap.fill(text, width=80))
    except Exception as e:
        print(text)  # fallback to normal print if text is not a string

        

load_dotenv('/Users/shivam13juna/Documents/scaler/iitr_classes/llm_ref/openai_key.env')  # reads .env file in the current directory

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError(
        "OPENAI_API_KEY not found! "
        "Make sure you have a .env file with: OPENAI_API_KEY=sk-..."
    )

pretty_print("API key loaded successfully.")
MODEL = "gpt-5-nano"


API key loaded successfully.


In [2]:
# Sanity check: one round trip through the Responses API.
response = client.responses.create(model=MODEL, input="Reply with exactly: API connection OK")
print(response.output_text)

API connection OK


In [3]:
# Limit 1 — no live information. The model has no clock, no sensors, no internet.
response = client.responses.create(
    model=MODEL,
    input="What is today's date, and what is the weather in Roorkee right now?",
    reasoning={"effort": "minimal"},   # or "none"
    text={"verbosity": "low"},
)
print(response.output_text)

I don’t have real-time access to current date/weather. 

- Today’s date: please check your device’s clock.
- Weather in Roorkee: you can check a weather app or website (e.g., weather.com, AccuWeather, Google Weather) for the latest conditions. If you’d like, I can guide you on how to find it quickly.


In [4]:
# Limit 2 — statelessness. Two SEPARATE API calls: tell it something, then ask for it back.
first = client.responses.create(
    model=MODEL,
    input="Hi! My name is Asha and I am allergic to penicillin. Please remember this.",
    reasoning={"effort": "minimal"},   # or "none"
    text={"verbosity": "low"},
)
print("Call 1:", first.output_text)

second = client.responses.create(
    model=MODEL,
    input="What am I allergic to?",
    reasoning={"effort": "minimal"},   # or "none"
    text={"verbosity": "low"},
)
print("\nCall 2:", second.output_text)

Call 1: Nice to meet you, Asha. I’ll remember that you’re allergic to penicillin. If you’d like, tell me any other allergies or important medical info I should keep in mind for future conversations.

Call 2: I can’t know without more information. Do you have a list of symptoms, triggers, or a device reading (like a test result)? A few clarifying questions:

- What symptoms do you have (hives, itching, swelling, sneezing, runny nose, asthma)?
- When did symptoms start and what were you exposed to recently?
- Any known allergies or family history?
- Have you had a recent allergy test (skin prick, blood test) or are you planning one?

If you’re experiencing severe symptoms (trouble breathing, swelling of face/throat, dizziness, or fainting), seek emergency care. If you want, tell me more details and I can help brainstorm possible allergens and next steps.


# The scenario: Riverside Family Clinic

**Riverside Family Clinic** is a small practice: two doctors, one receptionist, a few thousand registered patients. The clinic wants **MedAssist** — a virtual assistant that handles the front-desk workload that floods in every morning:

- *"Can I take X together with my usual medicines?"* (the single most common question)
- *"I need an appointment this week."*
- *"What did the doctor say about my dosage?"*
- *"Is this symptom urgent, or can it wait?"*

> **Asha Verma** — 58, patient ID **P-1001**. Atrial fibrillation and hypertension. Takes **warfarin** (a blood thinner with a famously narrow safety window) and amlodipine. Allergic to penicillin. Her doctor is Dr. Mulchand.
>
> This morning she messages the clinic:
>
> *"Hi, this is Asha Verma (patient ID P-1001). I've had a bad headache since yesterday. Can I just take ibuprofen for it? Also, please book me an appointment with my doctor this week if you think I should come in."*

1. Look up Asha's record. 
2. Judge any risk based on her history. 
3. If there are any red flags. 
4. Check calendar for Dr. Mulchand's availability.
5. Respond to Asha with a recommendation and an appointment if needed.
6. If Asha confirms then book appointment. 

# Part 1 Just the LLM

In [5]:
SYSTEM_RECEPTIONIST = """You are MedAssist, the virtual front-desk assistant for Riverside Family Clinic.
You help patients with medication questions and appointment booking."""

asha_message = """Hi, this is Asha Verma (patient ID P-1001). I've had a bad headache since yesterday.
Can I just take ibuprofen for it? Also, please book me an appointment with my doctor this week
if you think I should come in."""

response = client.responses.create(
    model=MODEL,
    input=[
        {"role": "system", "content": SYSTEM_RECEPTIONIST},
        {"role": "user", "content": asha_message}
    ],
        reasoning={"effort": "minimal"},   # or "none"
        text={"verbosity": "low"},
)
print(response.output_text)

Hi Asha. I’m sorry you’re not feeling well.

Regarding ibuprofen: I can give general info, but I can’t diagnose. For a headache since yesterday, taking an over-the-counter NSAID like ibuprofen is common for many people unless you have contraindications (e.g., GI ulcers, kidney disease, allergy, or if you’re on certain medications). Please confirm you don’t have any of the following:
- Pregnant or planning pregnancy
- History of stomach ulcers or kidney/liver issues
- Use of blood thinners or high blood pressure meds
- Allergies to NSAIDs

If none of those apply, you could follow the label directions. Take with food to reduce stomach upset. If the headache is severe, persistent beyond 48 hours, or accompanied by fever, stiff neck, confusion, or vomiting, seek urgent care.

Appointment booking:
- I can check Dr. [your doctor’s name]’s availability this week. Do you have a preferred day/time?
- Also relevant: is this for a physical/concierge visit or a specific concern (headache evaluatio

In [7]:
# Push directly on the action gap: demand the record lookup and a booking reference.
response = client.responses.create(
    model=MODEL,
    input=[
        {"role": "system", "content": SYSTEM_RECEPTIONIST},
        {"role": "user", "content": """Please look up my patient record (Asha Verma, P-1001), check whether
ibuprofen is safe with my current medications, then book the appointment and give me
the booking reference number."""},
    ],
        reasoning={"effort": "minimal"},   # or "none"
        text={"verbosity": "low"},
)
print(response.output_text)

I don’t have access to your patient records. I can’t verify drug interactions or book an appointment from here.

Here’s how we can proceed:
- Please confirm your current medications list (including dosages) or authorize me to pull it from your chart if you’re using a portal linked to Riverside Family Clinic.
- If you’d like, I can provide general information on ibuprofen interactions and safety warnings, but for a personalized check you’ll need to share or authorize access to your meds.

To book an appointment, tell me:
- Preferred date and time window
- Type of visit (general checkup, pain management, etc.)
- Any constraints or doctors you prefer

Alternatively, I can guide you on how to view your records and book online, or you can provide permission to fetch your records and I’ll proceed.


```
 you ──(1)──▶  "Here is my question, AND here is a menu of functions I promise
               to run for you if you ask. Here are their names and parameters."
 model ─(2)─▶  emits a `function_call` item: {name, arguments(JSON string), call_id}
               ── then STOPS. Nothing has happened in the world.
 you ──(3)──▶  read the item, json.loads the arguments, decide whether to honor it
 you ──(4)──▶  run the actual Python function yourself
 you ──(5)──▶  send back a `function_call_output` carrying the result + the same call_id
 model ─(6)─▶  continues, now treating your result as context it can read
```

In [8]:
interactions = pd.DataFrame(
    [
        # drug_a,        drug_b,            severity,   what happens,                                                                 standard recommendation,                                              source
        ("warfarin",     "ibuprofen",       "major",    "NSAIDs impair platelets and damage gastric mucosa; combined with an anticoagulant, risk of serious GI/intracranial bleeding rises sharply.", "Avoid. Prefer paracetamol for pain; any NSAID use needs prescriber approval.", "FDA warfarin label; AHA guidance"),
        ("warfarin",     "naproxen",        "major",    "Same NSAID bleeding mechanism as ibuprofen, longer-acting.",                  "Avoid; prescriber approval required.",                                "FDA warfarin label"),
        ("warfarin",     "aspirin",         "major",    "Antiplatelet effect adds to anticoagulation; bleeding risk increases substantially.", "Only with explicit cardiology/prescriber decision and monitoring.",   "FDA warfarin label"),
        ("warfarin",     "acetaminophen",   "moderate", "Regular use (≳2 g/day for several days) can potentiate warfarin and raise INR.", "Safest common analgesic on warfarin at occasional doses; check INR if used routinely.", "Hylek et al., JAMA 1998; NHS guidance"),
        ("warfarin",     "amoxicillin",     "moderate", "Antibiotics can disturb gut flora that produce vitamin K, raising INR.",      "Monitor INR more closely during and after the course.",               "FDA warfarin label"),
        ("warfarin",     "fluconazole",     "major",    "Potent CYP2C9 inhibition slows warfarin clearance; INR can spike dangerously.", "Avoid or reduce warfarin dose with close INR monitoring.",            "FDA fluconazole label"),
        ("lisinopril",   "ibuprofen",       "moderate", "NSAIDs blunt ACE-inhibitor effect and stress the kidneys (worse with a diuretic — the 'triple whammy').", "Avoid regular use; monitor BP and renal function if unavoidable.",     "FDA lisinopril label"),
        ("lisinopril",   "spironolactone",  "moderate", "Both raise potassium; together they risk hyperkalemia.",                      "Co-prescribe only with potassium monitoring.",                        "FDA labels (both)"),
        ("simvastatin",  "clarithromycin",  "major",    "Clarithromycin blocks CYP3A4, multiplying statin levels; risk of rhabdomyolysis (muscle breakdown).", "Contraindicated — pause the statin during the antibiotic course or switch antibiotic.", "FDA simvastatin label"),
        ("sertraline",   "tramadol",        "major",    "Both raise serotonin; combined use risks serotonin syndrome and lowers seizure threshold.", "Avoid; if unavoidable, lowest doses with close monitoring.",          "FDA tramadol label"),
        ("sertraline",   "ibuprofen",       "moderate", "SSRIs impair platelet aggregation; with NSAIDs, upper-GI bleeding risk roughly doubles.", "Prefer paracetamol; if NSAID needed, shortest course ± gastroprotection.", "FDA sertraline label; BMJ meta-analyses"),
        ("metformin",    "alcohol",         "moderate", "Heavy alcohol use with metformin raises the risk of lactic acidosis and hypoglycemia.", "Limit alcohol; avoid binge drinking.",                                "FDA metformin label"),
        ("levothyroxine","calcium carbonate","moderate", "Calcium binds levothyroxine in the gut and cuts its absorption.",            "Separate doses by at least 4 hours.",                                 "FDA levothyroxine label"),
        ("amlodipine",   "simvastatin",     "moderate", "Amlodipine raises simvastatin exposure; myopathy risk increases at high statin doses.", "Cap simvastatin at 20 mg/day when combined.",                         "FDA simvastatin label"),
    ],
    columns=["drug_a", "drug_b", "severity", "effect", "recommendation", "source"],
)

print(f"{len(interactions)} documented interactions in the clinic's table")
interactions[["drug_a", "drug_b", "severity", "recommendation"]]

14 documented interactions in the clinic's table


,drug_a,drug_b,severity,recommendation
0,warfarin,ibuprofen,major,Avoid. Prefer paracetamol for pain; any NSAID ...
1,warfarin,naproxen,major,Avoid; prescriber approval required.
2,warfarin,aspirin,major,Only with explicit cardiology/prescriber decis...
3,warfarin,acetaminophen,moderate,Safest common analgesic on warfarin at occasio...
4,warfarin,amoxicillin,moderate,Monitor INR more closely during and after the ...
5,warfarin,fluconazole,major,Avoid or reduce warfarin dose with close INR m...
6,lisinopril,ibuprofen,moderate,Avoid regular use; monitor BP and renal functi...
7,lisinopril,spironolactone,moderate,Co-prescribe only with potassium monitoring.
8,simvastatin,clarithromycin,major,Contraindicated — pause the statin during the ...
9,sertraline,tramadol,major,"Avoid; if unavoidable, lowest doses with close..."


In [11]:
## Live check against the real FDA label for warfarin (optional — needs internet; everything
## else in the notebook works offline). In production THIS is what a tool often is: an HTTP call.
#import urllib3
#import requests

#params = {"search": 'openfda.generic_name:"warfarin"', "limit": 1}
#try:
#    try:
#        resp = requests.get("https://api.fda.gov/drug/label.json", params=params, timeout=10)
#    except requests.exceptions.SSLError:
#        # Corporate VPNs/proxies often intercept TLS with their own certificate, which Python's
#        # cert bundle rejects. For a read-only public-API classroom demo we retry unverified;
#        # the production fix is installing the proxy's CA bundle — never ship verify=False.
#        urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
#        resp = requests.get("https://api.fda.gov/drug/label.json", params=params, timeout=10, verify=False)

#    label = resp.json()["results"][0]
#    pretty_print("From the live FDA label for acetaminophen:", label, "\n")
#    #print("From the live FDA label for warfarin:\n")
#    #pretty_print(label["drug_interactions"][0])
#except Exception as e:
#    print("openFDA unreachable right now — fine, the notebook doesn't depend on it.\n", repr(e))

# Part 2: LLM + Tool

In [12]:
check_interactions_tool = {
    "type": "function",
    "name": "check_drug_interactions",
    "description": (
        "Check for documented interactions between two or more drugs. "
        "Use this whenever a patient asks about combining medications, or before suggesting "
        "any new medication to a patient who already takes something. "
        "Accepts brand or generic names (e.g. 'Advil' or 'ibuprofen')."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "drugs": {
                "type": "array",
                "items": {"type": "string"},
                "description": "All drugs to check together, e.g. [\"warfarin\", \"ibuprofen\"]",
            }
        },
        "required": ["drugs"],
        "additionalProperties": False,
    },
    "strict": True,
}

BRAND_TO_GENERIC = {
    "advil": "ibuprofen", "motrin": "ibuprofen", "brufen": "ibuprofen", "nurofen": "ibuprofen",
    "tylenol": "acetaminophen", "paracetamol": "acetaminophen", "crocin": "acetaminophen", "calpol": "acetaminophen", "dolo": "acetaminophen",
    "coumadin": "warfarin", "jantoven": "warfarin",
    "aleve": "naproxen",
    "zoloft": "sertraline",
    "ultram": "tramadol",
    "zestril": "lisinopril", "prinivil": "lisinopril",
    "zocor": "simvastatin",
    "glucophage": "metformin",
    "norvasc": "amlodipine",
    "synthroid": "levothyroxine", "eltroxin": "levothyroxine",
    "ecosprin": "aspirin", "disprin": "aspirin",
}

def check_drug_interactions(drugs):
    """Check every pair among `drugs` against the clinic's interaction table."""
    names = sorted({BRAND_TO_GENERIC.get(d.strip().lower(), d.strip().lower()) for d in drugs})
    found = []
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            a, b = names[i], names[j]
            hits = interactions[
                ((interactions.drug_a == a) & (interactions.drug_b == b))
                | ((interactions.drug_a == b) & (interactions.drug_b == a))
            ]
            for row in hits.itertuples():
                found.append({
                    "pair": f"{row.drug_a} + {row.drug_b}",
                    "severity": row.severity,
                    "effect": row.effect,
                    "recommendation": row.recommendation,
                    "source": row.source,
                })
    return {
        "drugs_checked": names,
        "interactions_found": found,
        "note": "Teaching dataset — a small subset of documented interactions, not a complete reference.",
    }

result = check_drug_interactions(["warfarin", "Advil"])
print(json.dumps(result, indent=2))

{
  "drugs_checked": [
    "ibuprofen",
    "warfarin"
  ],
  "interactions_found": [
    {
      "pair": "warfarin + ibuprofen",
      "severity": "major",
      "effect": "NSAIDs impair platelets and damage gastric mucosa; combined with an anticoagulant, risk of serious GI/intracranial bleeding rises sharply.",
      "recommendation": "Avoid. Prefer paracetamol for pain; any NSAID use needs prescriber approval.",
      "source": "FDA warfarin label; AHA guidance"
    }
  ],
  "note": "Teaching dataset \u2014 a small subset of documented interactions, not a complete reference."
}


In [13]:
SYSTEM_RECEPTIONIST

'You are MedAssist, the virtual front-desk assistant for Riverside Family Clinic.\nYou help patients with medication questions and appointment booking.'

In [16]:
input_items = [
    {"role": "system", "content": SYSTEM_RECEPTIONIST},
    {"role": "user", "content": "Is it safe to take ibuprofen while I'm on warfarin? Please check properly before answering."},
]

response = client.responses.create(
    model=MODEL,
    input=input_items,
    reasoning={"effort": "minimal"},   # or "none"
    text={"verbosity": "low"},
    tools=[check_interactions_tool],
)

for item in response.output:
    print("output item type:", item.type)
    if item.type == "function_call":
        print("   name:      ", item.name)
        print("   arguments: ", repr(item.arguments), "   <-- a JSON *string* written by the model")
        print("   call_id:   ", item.call_id)

output item type: reasoning
output item type: function_call
   name:       check_drug_interactions
   arguments:  '{"drugs":["ibuprofen","warfarin"]}'    <-- a JSON *string* written by the model
   call_id:    call_fFxJb7UfvMezZLdkSTuUqej4


In [15]:
# The same round, using previous_response_id instead of a hand-managed list.
r1 = client.responses.create(
    model=MODEL,
    input=[
        {"role": "system", "content": SYSTEM_RECEPTIONIST},
        {"role": "user", "content": "Is it safe to take ibuprofen while I'm on warfarin? Please check properly."},
    ],
    reasoning={"effort": "minimal"},   # or "none"
    text={"verbosity": "low"},
    tools=[check_interactions_tool],
)

calls = [item for item in r1.output if item.type == "function_call"]
if not calls:
    print(r1.output_text)
else:
    tool_outputs = []
    for call in calls:
        result = check_drug_interactions(**json.loads(call.arguments))
        tool_outputs.append({"type": "function_call_output", "call_id": call.call_id, "output": json.dumps(result)})

    r2 = client.responses.create(
        model=MODEL,
        previous_response_id=r1.id,   # server replays the earlier turn; we send only the new tool results
        input=tool_outputs,
        reasoning={"effort": "minimal"},   # or "none"
        text={"verbosity": "low"},
        tools=[check_interactions_tool],
    )
    print(r2.output_text)

Short answer: not safe without doctor’s guidance.

What the check showed:
- Warfarin + ibuprofen (an NSAID) is a major interaction.
- Risks: increased bleeding (GI and other sites) and possible platelet/bleeding issues.
- Recommendation: Avoid ibuprofen while on warfarin. If you need pain relief, acetaminophen (paracetamol) is usually preferred, or your clinician may approve a different option. If NSAIDs are deemed necessary, it must be under a clinician’s explicit instruction with careful monitoring.

Next steps:
- Do not start or continue ibuprofen without talking to your clinician.
- If you’re experiencing pain, consider acetaminophen in the usual dose, unless you have liver issues or other contraindications.
- If you’ve already taken ibuprofen, monitor for unusual bleeding (unusual bruising, black/tarry stools, coughing up blood) and contact your clinician promptly.

Would you like me to alert your provider or help schedule a quick check-in?


| Decision | Who made it |
|---|---|
| How many times to call the model (exactly 2) | **You.** It's written in your cell structure. |
| That the tool result would be sent back, and then we'd stop | **You.** |
| What happens if the model wanted a *second* tool call after seeing the result | **Nothing — your code doesn't handle it.** The conversation simply ends. |
| What "done" means | **You** — done = "my script ran out of cells." |

# Part 3: LLM + Tool + Loop

In [17]:
PATIENT_DB = {
    "P-1001": {
        "name": "Asha Verma", "age": 58, "sex": "F",
        "conditions": ["atrial fibrillation", "hypertension"],
        "medications": [
            {"name": "warfarin",   "dose": "5 mg", "schedule": "once daily, evening"},
            {"name": "amlodipine", "dose": "5 mg", "schedule": "once daily, morning"},
        ],
        "allergies": ["penicillin"],
        "primary_doctor": "Dr. Mulchand",
        "last_visit": "2026-04-28",
        "clinical_notes": "Long-term anticoagulation for AF. INR monthly — last 2.6 (target 2.0–3.0).",
    },
    "P-1002": {
        "name": "Rohan Mehta", "age": 34, "sex": "M",
        "conditions": ["generalised anxiety disorder"],
        "medications": [
            {"name": "sertraline", "dose": "50 mg", "schedule": "once daily, morning"},
        ],
        "allergies": [],
        "primary_doctor": "Dr. Chomu",
        "last_visit": "2026-05-15",
        "clinical_notes": "Stable on sertraline since 2024.",
    },
    "P-1003": {
        "name": "Maria D'Souza", "age": 71, "sex": "F",
        "conditions": ["type 2 diabetes", "hypertension", "high cholesterol"],
        "medications": [
            {"name": "metformin",   "dose": "500 mg", "schedule": "twice daily, with meals"},
            {"name": "lisinopril",  "dose": "10 mg",  "schedule": "once daily, morning"},
            {"name": "simvastatin", "dose": "20 mg",  "schedule": "once daily, evening"},
        ],
        "allergies": ["sulfa drugs"],
        "primary_doctor": "Dr. Mulchand",
        "last_visit": "2026-06-02",
        "clinical_notes": "HbA1c 7.1%. Renal function normal at last check.",
    },
}

print(f"{len(PATIENT_DB)} patients in the toy EHR")

3 patients in the toy EHR


In [18]:
# The scheduling system: open slots for the next 3 working days, computed from *today*
# so this notebook stays runnable on any date.
upcoming_days = []
d = date.today()
while len(upcoming_days) < 3:
    d += timedelta(days=1)
    if d.weekday() < 5:                      # clinic closed Saturday/Sunday
        upcoming_days.append(d)

DOCTOR_SCHEDULE = {}
for doctor, times in [("Dr. Mulchand", ["09:30", "11:00", "15:30"]),
                      ("Dr. Chomu",    ["10:00", "14:00", "16:30"])]:
    for day in upcoming_days:
        for t in times:
            slot_id = f"{doctor.split()[-1].lower()}-{day.isoformat()}-{t.replace(':', '')}"
            DOCTOR_SCHEDULE[slot_id] = {
                "doctor": doctor, "date": day.isoformat(),
                "day": day.strftime("%A"), "time": t, "status": "open",
            }

APPOINTMENT_BOOK = []     # real side effects will land here

print(f"{len(DOCTOR_SCHEDULE)} open slots over the next 3 working days\n")
pd.DataFrame(DOCTOR_SCHEDULE).T.head(6)

18 open slots over the next 3 working days



,doctor,date,day,time,status
mulchand-2026-06-12-0930,Dr. Mulchand,2026-06-12,Friday,09:30,open
mulchand-2026-06-12-1100,Dr. Mulchand,2026-06-12,Friday,11:00,open
mulchand-2026-06-12-1530,Dr. Mulchand,2026-06-12,Friday,15:30,open
mulchand-2026-06-15-0930,Dr. Mulchand,2026-06-15,Monday,09:30,open
mulchand-2026-06-15-1100,Dr. Mulchand,2026-06-15,Monday,11:00,open
mulchand-2026-06-15-1530,Dr. Mulchand,2026-06-15,Monday,15:30,open


1. Agent to be able to retrieve patient-records
2. Agent to be able to check doctor's calendar
3. Agent to be able to book appointments. 

In [19]:
def get_patient_record(patient_id):
    """Fetch a patient's record from the clinic EHR (toy stand-in for a FHIR API)."""
    record = PATIENT_DB.get(patient_id.strip().upper())
    if record is None:
        return {"error": f"No patient found with ID {patient_id!r}. Ask the patient to double-check it."}
    return record

def get_doctor_availability(doctor=None):
    """List open slots — all doctors, or filtered to one."""
    slots = [
        {"slot_id": sid, **info}
        for sid, info in DOCTOR_SCHEDULE.items()
        if info["status"] == "open" and (doctor is None or doctor.lower() in info["doctor"].lower())
    ]
    if not slots:
        return {"error": f"No open slots matching doctor={doctor!r}.",
                "hint": "Call again with doctor=null to see all doctors."}
    return {"open_slots": slots}

def book_appointment(patient_id, slot_id, reason):
    """Book a slot. THIS HAS A REAL SIDE EFFECT on clinic state — note all the validation."""
    pid = patient_id.strip().upper()
    if pid not in PATIENT_DB:
        return {"error": f"Unknown patient ID {patient_id!r} — refusing to book."}
    slot = DOCTOR_SCHEDULE.get(slot_id)
    if slot is None:
        return {"error": f"Slot {slot_id!r} does not exist. Use get_doctor_availability for valid slot_ids."}
    if slot["status"] != "open":
        return {"error": f"Slot {slot_id!r} is already taken. Pick another open slot."}

    slot["status"] = "booked"
    confirmation = {
        "confirmation_id": f"APT-{len(APPOINTMENT_BOOK) + 1:04d}",
        "patient_id": pid,
        "patient_name": PATIENT_DB[pid]["name"],
        "doctor": slot["doctor"], "day": slot["day"], "date": slot["date"], "time": slot["time"],
        "reason": reason,
    }
    APPOINTMENT_BOOK.append(confirmation)
    return {"status": "confirmed", **confirmation}

In [20]:
AGENT_TOOLS = [
    check_interactions_tool,          # from Part 4
    {
        "type": "function",
        "name": "get_patient_record",
        "description": (
            "Fetch a registered patient's medical record: conditions, current medications, allergies, "
            "primary doctor, recent notes. Always fetch this BEFORE giving medication or booking advice "
            "to an identified patient."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "patient_id": {"type": "string", "description": "Clinic patient ID, e.g. 'P-1001'."},
            },
            "required": ["patient_id"],
            "additionalProperties": False,
        },
        "strict": True,
    },
    {
        "type": "function",
        "name": "get_doctor_availability",
        "description": "List open appointment slots for the next few working days, with their slot_ids.",
        "parameters": {
            "type": "object",
            "properties": {
                "doctor": {
                    "type": ["string", "null"],
                    "description": "Doctor name to filter by (e.g. 'Dr. Mulchand'), or null for all doctors.",
                },
            },
            "required": ["doctor"],                # strict mode: optional == required-but-nullable
            "additionalProperties": False,
        },
        "strict": True,
    },
    {
        "type": "function",
        "name": "book_appointment",
        "description": (
            "Book an appointment slot for a patient. Irreversible in this system — only call after "
            "choosing a specific open slot_id from get_doctor_availability."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "patient_id": {"type": "string", "description": "Clinic patient ID."},
                "slot_id": {"type": "string", "description": "Exact slot_id from get_doctor_availability."},
                "reason": {"type": "string", "description": "Short reason for the visit, for the doctor's notes."},
            },
            "required": ["patient_id", "slot_id", "reason"],
            "additionalProperties": False,
        },
        "strict": True,
    },
]

TOOL_REGISTRY = {
    "check_drug_interactions": check_drug_interactions,
    "get_patient_record": get_patient_record,
    "get_doctor_availability": get_doctor_availability,
    "book_appointment": book_appointment,
}

print("Agent can use:", ", ".join(TOOL_REGISTRY))

Agent can use: check_drug_interactions, get_patient_record, get_doctor_availability, book_appointment


Each function is a tool the agent can call. The agent can call as many as it wants, in any order, and use the results to inform its next steps. The agent decides when it's done, and what "done" means.

In [21]:
AGENT_SYSTEM_PROMPT = f"""You are MedAssist, the virtual assistant for Riverside Family Clinic.
Today is {date.today().strftime('%A, %d %B %Y')}.

You have tools to read patient records, check drug interactions, view doctor availability,
and book appointments. Follow these rules:

1. When a patient gives their patient ID, fetch their record BEFORE giving any medication
   or booking advice.
2. Never answer medication-combination questions from your own knowledge. Always call
   check_drug_interactions with the patient's current medications plus the proposed drug.
3. If the patient should be seen and asks you to book, book the earliest suitable slot with
   their primary doctor, then confirm doctor, day, date, time and confirmation ID in your reply.
4. If symptoms suggest an emergency (sudden 'worst-ever' headache, stroke signs, chest pain,
   uncontrolled bleeding), tell the patient to seek emergency care immediately instead of booking.
5. Be warm, concise, plain-spoken. You are not a doctor: frame advice as guidance and defer
   to clinicians for decisions.
"""
print(AGENT_SYSTEM_PROMPT)

You are MedAssist, the virtual assistant for Riverside Family Clinic.
Today is Thursday, 11 June 2026.

You have tools to read patient records, check drug interactions, view doctor availability,
and book appointments. Follow these rules:

1. When a patient gives their patient ID, fetch their record BEFORE giving any medication
   or booking advice.
2. Never answer medication-combination questions from your own knowledge. Always call
   check_drug_interactions with the patient's current medications plus the proposed drug.
3. If the patient should be seen and asks you to book, book the earliest suitable slot with
   their primary doctor, then confirm doctor, day, date, time and confirmation ID in your reply.
4. If symptoms suggest an emergency (sudden 'worst-ever' headache, stroke signs, chest pain,
   uncontrolled bleeding), tell the patient to seek emergency care immediately instead of booking.
5. Be warm, concise, plain-spoken. You are not a doctor: frame advice as guidance and defer


In [22]:
def run_agent(user_message, history=None, max_iterations=8, verbose=True):
    """A minimal ReAct agent on the Responses API.

    One *iteration* = one model call. In each iteration the model either
    (a) requests tool calls -> we execute them and loop again, or
    (b) writes plain text   -> that's its final answer; we stop.
    """
    if history is None:
        history = [{"role": "system", "content": AGENT_SYSTEM_PROMPT}]
    history.append({"role": "user", "content": user_message})

    for iteration in range(1, max_iterations + 1):

        # ---- THINK: the model sees everything so far and decides what to do next
        response = client.responses.create(model=MODEL, input=history, tools=AGENT_TOOLS)
        history += response.output                      # its thoughts/requests join the transcript

        tool_calls = [item for item in response.output if item.type == "function_call"]

        # ---- DONE? No tool request means the model decided the goal is met.
        if not tool_calls:
            if verbose:
                print(f"\n[iteration {iteration}] ✅ final answer\n" + "-" * 60)
            return response.output_text, history

        # ---- ACT + OBSERVE: execute every requested call, feed results back
        for call in tool_calls:
            args = json.loads(call.arguments)
            if verbose:
                print(f"[iteration {iteration}] 🔧 {call.name}({call.arguments})")
            result = TOOL_REGISTRY[call.name](**args)
            payload = json.dumps(result, default=str)
            if verbose:
                print(f"              ↳ {payload[:150]}{'…' if len(payload) > 150 else ''}")
            history.append({
                "type": "function_call_output",
                "call_id": call.call_id,
                "output": payload,
            })

    # Loop budget exhausted — Part 6 is about doing this gracefully.
    return "Stopped: hit max_iterations without a final answer.", history

In [23]:
final_answer, asha_history = run_agent(
    """Hi, this is Asha Verma (patient ID P-1001). I've had a bad headache since yesterday.
Can I just take ibuprofen for it? Also, please book me an appointment with my doctor this week, but do confirm with me the slots before booking
if you think I should come in."""
)
print(final_answer)

[iteration 1] 🔧 get_patient_record({"patient_id":"P-1001"})
              ↳ {"name": "Asha Verma", "age": 58, "sex": "F", "conditions": ["atrial fibrillation", "hypertension"], "medications": [{"name": "warfarin", "dose": "5 m…
[iteration 2] 🔧 check_drug_interactions({"drugs":["warfarin","amlodipine","ibuprofen"]})
              ↳ {"drugs_checked": ["amlodipine", "ibuprofen", "warfarin"], "interactions_found": [{"pair": "warfarin + ibuprofen", "severity": "major", "effect": "NSA…
[iteration 3] 🔧 get_doctor_availability({"doctor":"Dr. Mulchand"})
              ↳ {"open_slots": [{"slot_id": "mulchand-2026-06-12-0930", "doctor": "Dr. Mulchand", "date": "2026-06-12", "day": "Friday", "time": "09:30", "status": "o…

[iteration 4] ✅ final answer
------------------------------------------------------------
Hi Asha — thanks for the details. I pulled up your chart and here’s what I found and what I suggest.

Medication and ibuprofen
- Current meds: warfarin (5 mg, nightly) and amlodipine (5 mg,

In [24]:
final_answer_v2, asha_history_v2 = run_agent(
    """Are you sure about the drug interaction? And do you have any latest slots available? If not, book me the first slot available.""", history=asha_history
)
print(final_answer_v2)


[iteration 1] ✅ final answer
------------------------------------------------------------
Thanks for checking. Here’s the update:

Drug interaction
- Yes, the check shows a major interaction: warfarin + ibuprofen. NSAIDs can increase bleeding risk when you’re on an anticoagulant, including warfarin.
- Safer option for pain relief while on warfarin is acetaminophen (paracetamol), taken per product instructions and without liver issues. If you have liver disease, heavy alcohol use, or are unsure about dosing, please check with a clinician before taking it regularly.
- If you have any signs of unusual bleeding (e.g., very dark stools, vomiting blood) or severe headache with new neurological symptoms, seek medical care promptly.

Latest available doctor slots with Dr. Mulchand
- Tue 2026-06-16 at 15:30 — slot_id mulchand-2026-06-16-1530 (latest in the list)
- Other open slots this week:
  - Fri 2026-06-12 09:30 — slot_id mulchand-2026-06-12-0930
  - Fri 2026-06-12 11:00 — slot_id mulchand

In [25]:
final_answer_v3, asha_history_v3 = run_agent(
    """Yes, please look me the first slot available..""", history=asha_history_v2
)
print(final_answer_v3)

[iteration 1] 🔧 book_appointment({"patient_id":"P-1001","slot_id":"mulchand-2026-06-12-0930","reason":"Headache evaluation and management discussion (review meds, given warfarin)."})
              ↳ {"status": "confirmed", "confirmation_id": "APT-0001", "patient_id": "P-1001", "patient_name": "Asha Verma", "doctor": "Dr. Mulchand", "day": "Friday"…

[iteration 2] ✅ final answer
------------------------------------------------------------
You’re all set, Asha.

Appointment booked
- Doctor: Dr. Mulchand
- Date: Friday, 2026-06-12
- Time: 09:30
- Reason: Headache evaluation and management discussion (review meds, given warfarin)
- Confirmation ID: APT-0001

Notes
- You mentioned a headache since yesterday. We’ll review your current meds (warfarin and amlodipine) and discuss safe pain relief options (paracetamol) and any red flags to watch for.
- If you experience new or worsening symptoms before the visit (e.g., chest pain, sudden severe headache, weakness/numbness on one side, trouble sp